# scETM Baseline for Single-Cell RNA-seq Analysis

This notebook provides a baseline implementation using scETM (single-cell Embedded Topic Model) for interpretable single-cell RNA-seq data analysis.

## 📋 Prerequisites

Before running this baseline, please ensure you have:

1. **Followed the main installation guide** in the root directory's README to set up the conda environment
2. **Installed additional dependencies** specific to scETM:

```bash
# Activate the scE2TM environment first
conda activate scE2TM_env

# Install scETM and its dependencies
pip install scETM

In [ ]:
"""
scETM baseline implementation for single-cell RNA-seq analysis
This script performs topic modeling, dimensionality reduction, clustering, and evaluation using scETM
"""

import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, silhouette_score
from sklearn import metrics
import os
import sys
import torch
from scETM import scETM, UnsupervisedTrainer, evaluate, prepare_for_transfer

# Get the current directory - works in both Jupyter notebook and Python scripts
try:
    # This works when running as a script
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # This works in Jupyter notebook
    current_dir = os.getcwd()

print(f"Current working directory: {current_dir}")

# Define data paths (relative to the script location)
data_name = 'Wang'  # Base name for the data files
data_path = os.path.join(current_dir, '..', 'data', f'{data_name}_HIGHPRE.csv')  # Expression data
label_path = os.path.join(current_dir, '..', 'data', f'{data_name}_cell_anno.csv')  # Cell type annotations

# Define output directory (create if it doesn't exist)
output_dir = os.path.join(current_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print(f"Loading data from: {data_path}")
print(f"Loading labels from: {label_path}")
print(f"Outputs will be saved to: {output_dir}")

# Check if files exist
if not os.path.exists(data_path):
    print(f"ERROR: Data file not found at {data_path}")
    print("Please make sure the data file is in the correct location.")
    print("Expected structure:")
    print("  scE2TM/")
    print("  ├── baseline/")
    print("  │   ├── scETM.ipynb (this notebook)")
    print("  │   └── output/ (will be created)")
    print("  └── data/")
    print("      ├── Wang_HIGHPRE.csv")
    print("      └── Wang_cell_anno.csv")
    sys.exit(1)

if not os.path.exists(label_path):
    print(f"ERROR: Label file not found at {label_path}")
    print("Please make sure the label file is in the correct location.")
    sys.exit(1)

# Load expression data
# The data should be in CSV format with genes as columns and cells as rows
scETM_csv = pd.read_csv(data_path, sep=',', index_col=0)
print(f"Data shape: {scETM_csv.shape}")

# Create AnnData object (standard format for single-cell data)
scETM_adata = sc.AnnData(scETM_csv)

# Load cell type labels
# These are the ground truth labels for evaluation
scETM_label_csv = pd.read_csv(label_path, sep=',', index_col=0)
# Extract the first column as cell type labels
scETM_adata.obs['cell_type'] = list(scETM_label_csv.iloc[:, 0])
print(f"Number of cell types: {len(scETM_adata.obs['cell_type'].unique())}")

# Handle potential NaN values in labels
non_nan_indices = [i for i, x in enumerate(scETM_adata.obs['cell_type']) if pd.notna(x)]
print(f"Cells with valid labels: {len(non_nan_indices)}/{len(scETM_adata.obs['cell_type'])}")

# Add batch indices (required by scETM, here we use all zeros since no batch effect)
scETM_adata.obs['batch_indices'] = np.zeros(len(scETM_adata), dtype=int)

# Initialize scETM model
# n_topics: number of topics (latent dimensions)
n_topics = 100
print(f"Initializing scETM model with {n_topics} topics...")

#根据scETM源码，参数名可能是n_vars而不是n_genes
model = scETM(
    scETM_adata.n_vars,  # 第一个参数通常是n_genes/n_vars
    1,  # n_batches
    device=torch.device("cuda:0"),  # Use GPU 0 for training
    n_topics=n_topics  # 明确指定主题数
)

# Setup trainer
trainer = UnsupervisedTrainer(
    model, 
    scETM_adata, 
    test_ratio=0  # Use all data for training
)

# Train model
print("Training scETM model...")
trainer.train(
    n_epochs=12000, 
    eval_every=3000, 
    eval_kwargs=dict(cell_type_col='cell_type'),
    ckpt_dir=output_dir  # Save checkpoints to output directory
)

# Extract latent representation (cell-topic distribution: delta)
# This is the cell embedding in topic space
latent = scETM_adata.obsm["delta"]  # Shape: (n_cells, n_topics)
print(f"Latent representation shape: {latent.shape}")

# Extract topic-gene distribution (beta)
# This represents the relationship between topics and genes
topic_gene = model.rho.detach().cpu().numpy()  # Shape: (n_topics, n_genes)
print(f"Topic-gene matrix shape: {topic_gene.shape}")

# Extract topic embeddings (alpha)
# This represents the topic embeddings in the latent space
topic_embedding = model.alpha.detach().cpu().numpy()  # Shape: (n_topics, embedding_dim)
print(f"Topic embedding shape: {topic_embedding.shape}")

# Construct neighborhood graph using scETM latent representation
sc.pp.neighbors(scETM_adata, n_neighbors=15, use_rep='delta')

# Find optimal resolution for Louvain clustering
# We search over a range of resolutions to maximize ARI
max_resolution = 2  # Maximum resolution to test
min_resolution = 0  # Minimum resolution to test
n_steps = 20  # Number of resolution steps (max_resolution * 10)
ari_scores = []

print("Searching for optimal clustering resolution...")
for i in range(min_resolution, n_steps):
    resolution = i / 10.0
    sc.tl.louvain(scETM_adata, resolution=resolution, random_state=0)
    # Only use cells with valid labels for ARI calculation
    ari = adjusted_rand_score(
        scETM_adata.obs['cell_type'][non_nan_indices], 
        scETM_adata.obs['louvain'][non_nan_indices]
    )
    ari_scores.append(ari)
    print(f"Resolution {resolution:.1f}: ARI = {ari:.4f}")

# Select the resolution that gives the highest ARI
best_resolution = ari_scores.index(max(ari_scores)) * 0.1
print(f"Best resolution: {best_resolution:.1f} (ARI = {max(ari_scores):.4f})")

# Perform final clustering with optimal resolution
sc.tl.louvain(scETM_adata, resolution=best_resolution, random_state=0)

# Compute UMAP for visualization
sc.tl.umap(scETM_adata)

# Visualize results
sc.pl.umap(
    scETM_adata,
    color=["cell_type", "louvain"],
    wspace=0.3,
    title=['Ground Truth Cell Types', 'scETM Clustering Results']
    # Uncomment the line below to save the figure
    # save=os.path.join(output_dir, f'scETM_{data_name}_umap.pdf')
)

# Calculate and print evaluation metrics
ari = adjusted_rand_score(
    scETM_adata.obs['cell_type'][non_nan_indices], 
    scETM_adata.obs['louvain'][non_nan_indices]
)
ami = adjusted_mutual_info_score(
    scETM_adata.obs['cell_type'][non_nan_indices], 
    scETM_adata.obs['louvain'][non_nan_indices]
)
asw = silhouette_score(
    scETM_adata.X[non_nan_indices],  # Using original gene expression for ASW
    scETM_adata.obs['cell_type'][non_nan_indices]
)

print("\n" + "="*50)
print("EVALUATION METRICS")
print("="*50)
print(f"Adjusted Rand Index (ARI):      {ari:.4f}")
print(f"Adjusted Mutual Information (AMI): {ami:.4f}")
print(f"Average Silhouette Width (ASW): {asw:.4f}")
print("="*50)

# Calculate purity score based on dominant topic
dominant_topics = np.argmax(latent, axis=1)

def purity_score(y_true, y_pred):
    """
    Calculate purity score for clustering evaluation
    Purity measures the extent to which each cluster contains only cells from a single class
    """
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

purity = purity_score(
    scETM_adata.obs['cell_type'][non_nan_indices], 
    dominant_topics[non_nan_indices]
)
print(f"Purity Score (based on dominant topic): {purity:.4f}")
print("="*50)

# ============================================================
# SAVE OUTPUTS
# ============================================================
print("\n" + "="*50)
print("SAVING OUTPUTS")
print("="*50)
print(f"All outputs will be saved to: {output_dir}")

# Save topic-gene matrix (beta)
topic_gene_path = os.path.join(output_dir, f'scETM_{data_name}_topic_gene.csv')
pd.DataFrame(topic_gene,
             index=[f'Topic_{i+1}' for i in range(topic_gene.shape[0])],
             columns=scETM_adata.var_names).to_csv(topic_gene_path)
print(f"✓ Topic-gene matrix saved: scETM_{data_name}_topic_gene.csv")

# Save cell embeddings (delta)
embedding_path = os.path.join(output_dir, f'scETM_{data_name}_embedding.csv')
pd.DataFrame(latent,
             index=scETM_adata.obs_names,
             columns=[f'Topic_{i+1}' for i in range(latent.shape[1])]).to_csv(embedding_path)
print(f"✓ Cell embeddings saved: scETM_{data_name}_embedding.csv")

# Save topic embeddings (alpha)
topic_embedding_path = os.path.join(output_dir, f'scETM_{data_name}_topic_embedding.csv')
pd.DataFrame(topic_embedding,
             index=[f'Topic_{i+1}' for i in range(topic_embedding.shape[0])],
             columns=[f'Dim_{i+1}' for i in range(topic_embedding.shape[1])]).to_csv(topic_embedding_path)
print(f"✓ Topic embeddings saved: scETM_{data_name}_topic_embedding.csv")

# Save gene embeddings (rho) - this is the same as topic_gene matrix transposed
gene_embedding_path = os.path.join(output_dir, f'scETM_{data_name}_gene_embedding.csv')
pd.DataFrame(topic_gene.T,
             index=scETM_adata.var_names,
             columns=[f'Topic_{i+1}' for i in range(topic_gene.shape[0])]).to_csv(gene_embedding_path)
print(f"✓ Gene embeddings saved: scETM_{data_name}_gene_embedding.csv")

print("="*50)
print("DONE!")
print("="*50)